The 80/20 split is fixed before modelling, stratified by `ClaimNb > 0` with `random_state=42`. Model fitting and selection use only the train split; the holdout is used afterward for evaluation and error analysis. `Exposure` is a sample weight, never a feature.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display
from lightgbm import LGBMRegressor
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import mean_poisson_deviance
from sklearn.model_selection import ParameterGrid, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, SplineTransformer, StandardScaler

claims = pd.read_parquet(Path("data") / "claims_prepared.parquet")
feature_columns = ["Area", "VehPower", "VehAge", "DrivAge", "BonusMalus", "VehBrand", "VehGas", "Density", "Region"]
train_index, test_index = train_test_split(
    claims.index,
    test_size=0.20,
    stratify=claims["ClaimNb"].gt(0),
    random_state=42,
)
train = claims.loc[train_index]
X_train = train[feature_columns]
y_train = train["ClaimNb"] / train["Exposure"]
weight_train = train["Exposure"]
strata_train = train["ClaimNb"].gt(0)


In [2]:
baseline_rate = train["ClaimNb"].sum() / train["Exposure"].sum()
assert np.isclose(baseline_rate, 0.100617907158, atol=5e-13, rtol=0.0)
baseline_rate


np.float64(0.10061790715750009)

In [3]:
def cv_search(pipeline, parameter_grid, X, y, weight, strata, cv):
    rows = []
    for parameters in ParameterGrid(parameter_grid):
        fold_scores = []
        for fit_index, validation_index in cv.split(X, strata):
            candidate = clone(pipeline).set_params(**parameters)
            candidate.fit(
                X.iloc[fit_index],
                y.iloc[fit_index],
                model__sample_weight=weight.iloc[fit_index].to_numpy(),
            )
            prediction = candidate.predict(X.iloc[validation_index])
            fold_scores.append(
                mean_poisson_deviance(
                    y.iloc[validation_index],
                    prediction,
                    sample_weight=weight.iloc[validation_index],
                )
            )
        rows.append(
            {
                "parameters": parameters,
                "cv_mean_deviance": np.mean(fold_scores),
                "cv_std_deviance": np.std(fold_scores, ddof=1),
            }
        )
    return pd.DataFrame(rows).sort_values("cv_mean_deviance").reset_index(drop=True)


def calibration_deciles(model_name, actual_claims, exposure, prediction):
    ordered = pd.DataFrame(
        {"actual_claims": actual_claims, "exposure": exposure, "prediction": prediction}
    ).sort_values("prediction", kind="mergesort")
    exposure_position = (ordered["exposure"].cumsum() - 0.5 * ordered["exposure"]) / ordered["exposure"].sum()
    ordered["risk_decile"] = np.clip(np.floor(10 * exposure_position).astype(int) + 1, 1, 10)
    ordered["predicted_claims"] = ordered["exposure"] * ordered["prediction"]
    result = ordered.groupby("risk_decile").agg(
        exposure=("exposure", "sum"),
        actual_claims=("actual_claims", "sum"),
        predicted_claims=("predicted_claims", "sum"),
    )
    result["observed_frequency"] = result["actual_claims"] / result["exposure"]
    result["predicted_frequency"] = result["predicted_claims"] / result["exposure"]
    result["lift"] = result["observed_frequency"] / (ordered["actual_claims"].sum() / ordered["exposure"].sum())
    return result.reset_index().assign(model=model_name)[
        ["model", "risk_decile", "exposure", "actual_claims", "predicted_claims", "observed_frequency", "predicted_frequency", "lift"]
    ]


def weighted_gini(actual_claims, exposure, score):
    ranked = pd.DataFrame(
        {"actual_claims": actual_claims, "exposure": exposure, "score": score}
    ).groupby("score", as_index=False).agg(actual_claims=("actual_claims", "sum"), exposure=("exposure", "sum"))
    ranked = ranked.sort_values("score", ascending=False)
    exposure_share = np.r_[0.0, ranked["exposure"].cumsum() / ranked["exposure"].sum()]
    claims_share = np.r_[0.0, ranked["actual_claims"].cumsum() / ranked["actual_claims"].sum()]
    return 2.0 * np.trapezoid(claims_share, exposure_share) - 1.0


The GLM provides a transparent Poisson benchmark, while LightGBM captures nonlinearities and interactions. Two complementary models are enough for the brief.

In [4]:
smooth_features = ["DrivAge", "VehAge", "BonusMalus"]
glm_categorical_features = ["Area", "VehPower", "VehBrand", "VehGas", "Region"]
lgbm_categorical_features = ["Area", "VehBrand", "VehGas", "Region"]
lgbm_ordinal_features = ["VehPower", "DrivAge", "VehAge", "BonusMalus"]
log_density = Pipeline(
    [
        ("log", FunctionTransformer(np.log, feature_names_out="one-to-one")),
        ("scale", StandardScaler()),
    ]
)
glm_preprocess = ColumnTransformer(
    [
        ("splines", SplineTransformer(degree=3, knots="quantile", include_bias=False, extrapolation="linear"), smooth_features),
        ("categories", OneHotEncoder(handle_unknown="ignore", drop="first"), glm_categorical_features),
        ("log_density", log_density, ["Density"]),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)
lgbm_preprocess = ColumnTransformer(
    [
        ("ordinal", "passthrough", lgbm_ordinal_features),
        ("categories", OneHotEncoder(handle_unknown="ignore"), lgbm_categorical_features),
        ("log_density", FunctionTransformer(np.log, feature_names_out="one-to-one"), ["Density"]),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)
glm_pipeline = Pipeline(
    [
        ("preprocess", glm_preprocess),
        ("model", PoissonRegressor(max_iter=1000)),
    ]
)
lgbm_pipeline = Pipeline(
    [
        ("preprocess", lgbm_preprocess),
        ("model", LGBMRegressor(objective="poisson", random_state=42, n_jobs=-1, verbosity=-1, deterministic=True, force_col_wise=True)),
    ]
)
glm_grid = {
    "preprocess__splines__n_knots": [4, 6, 8],
    "model__alpha": [0.0, 1e-6, 1e-4, 1e-2],
}
lgbm_grid = {
    "model__learning_rate": [0.03, 0.06],
    "model__n_estimators": [300, 600],
    "model__num_leaves": [15, 31],
    "model__min_child_samples": [100, 500],
}
search_space = pd.DataFrame(
    [
        {"model": "GLM", "parameter": parameter, "values": values}
        for parameter, values in glm_grid.items()
    ]
    + [
        {"model": "LightGBM", "parameter": parameter, "values": values}
        for parameter, values in lgbm_grid.items()
    ]
)
display(search_space)


,model,parameter,values
0,GLM,preprocess__splines__n_knots,"[4, 6, 8]"
1,GLM,model__alpha,"[0.0, 1e-06, 0.0001, 0.01]"
2,LightGBM,model__learning_rate,"[0.03, 0.06]"
3,LightGBM,model__n_estimators,"[300, 600]"
4,LightGBM,model__num_leaves,"[15, 31]"
5,LightGBM,model__min_child_samples,"[100, 500]"


The GLM grid varies spline knots and regularisation; the LightGBM grid varies learning rate, trees, leaves and minimum child size. The grids are deliberately small. LightGBM selects the edge of the grid, but the search is not expanded after opening the test set.

In [5]:
glm_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
glm_cv_results = cv_search(
    glm_pipeline, glm_grid, X_train, y_train, weight_train, strata_train, glm_cv
)


In [6]:
glm_best_parameters = glm_cv_results.loc[0, "parameters"]
glm_model = clone(glm_pipeline).set_params(**glm_best_parameters)
glm_model.fit(X_train, y_train, model__sample_weight=weight_train.to_numpy())
display(glm_cv_results)
glm_best_parameters


,parameters,cv_mean_deviance,cv_std_deviance
0,"{'model__alpha': 0.0, 'preprocess__splines__n_...",0.593327,0.002953
1,"{'model__alpha': 1e-06, 'preprocess__splines__...",0.593354,0.002979
2,"{'model__alpha': 0.0, 'preprocess__splines__n_...",0.593440,0.003255
3,"{'model__alpha': 1e-06, 'preprocess__splines__...",0.593455,0.003195
4,"{'model__alpha': 0.0001, 'preprocess__splines_...",0.594137,0.002840
5,"{'model__alpha': 0.0, 'preprocess__splines__n_...",0.594182,0.002946
6,"{'model__alpha': 1e-06, 'preprocess__splines__...",0.594216,0.002919
7,"{'model__alpha': 0.0001, 'preprocess__splines_...",0.595378,0.002596
8,"{'model__alpha': 0.0001, 'preprocess__splines_...",0.597294,0.002375
9,"{'model__alpha': 0.01, 'preprocess__splines__n...",0.606786,0.002541


{'model__alpha': 0.0, 'preprocess__splines__n_knots': 6}

In [7]:
lgbm_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
lgbm_cv_results = cv_search(
    lgbm_pipeline, lgbm_grid, X_train, y_train, weight_train, strata_train, lgbm_cv
)


In [8]:
lgbm_best_parameters = lgbm_cv_results.loc[0, "parameters"]
lgbm_model = clone(lgbm_pipeline).set_params(**lgbm_best_parameters)
lgbm_model.fit(X_train, y_train, model__sample_weight=weight_train.to_numpy())
display(lgbm_cv_results)
lgbm_best_parameters


,parameters,cv_mean_deviance,cv_std_deviance
0,"{'model__learning_rate': 0.06, 'model__min_chi...",0.573412,0.001548
1,"{'model__learning_rate': 0.06, 'model__min_chi...",0.573612,0.002062
2,"{'model__learning_rate': 0.03, 'model__min_chi...",0.573614,0.001785
3,"{'model__learning_rate': 0.06, 'model__min_chi...",0.573665,0.001713
4,"{'model__learning_rate': 0.03, 'model__min_chi...",0.574035,0.002299
5,"{'model__learning_rate': 0.06, 'model__min_chi...",0.574102,0.002329
6,"{'model__learning_rate': 0.03, 'model__min_chi...",0.574543,0.001891
7,"{'model__learning_rate': 0.06, 'model__min_chi...",0.574634,0.001928
8,"{'model__learning_rate': 0.06, 'model__min_chi...",0.574885,0.002134
9,"{'model__learning_rate': 0.03, 'model__min_chi...",0.575071,0.002297


{'model__learning_rate': 0.06,
 'model__min_child_samples': 100,
 'model__n_estimators': 600,
 'model__num_leaves': 31}

In [9]:
test = claims.loc[test_index]
X_test = test[feature_columns]
y_test = test["ClaimNb"] / test["Exposure"]
weight_test = test["Exposure"]
glm_train_prediction = glm_model.predict(X_train)
lgbm_train_prediction = lgbm_model.predict(X_train)
glm_test_prediction = glm_model.predict(X_test)
lgbm_test_prediction = lgbm_model.predict(X_test)
baseline_train_prediction = np.full(len(train), baseline_rate)
baseline_test_prediction = np.full(len(test), baseline_rate)
deviance_table = pd.DataFrame(
    {
        "model": ["Baseline", "GLM", "LightGBM"],
        "train_mean_poisson_deviance": [
            mean_poisson_deviance(y_train, baseline_train_prediction, sample_weight=weight_train),
            mean_poisson_deviance(y_train, glm_train_prediction, sample_weight=weight_train),
            mean_poisson_deviance(y_train, lgbm_train_prediction, sample_weight=weight_train),
        ],
        "test_mean_poisson_deviance": [
            mean_poisson_deviance(y_test, baseline_test_prediction, sample_weight=weight_test),
            mean_poisson_deviance(y_test, glm_test_prediction, sample_weight=weight_test),
            mean_poisson_deviance(y_test, lgbm_test_prediction, sample_weight=weight_test),
        ],
    }
)
total_calibration = pd.DataFrame(
    {
        "model": ["GLM", "LightGBM"],
        "actual_claims": [test["ClaimNb"].sum(), test["ClaimNb"].sum()],
        "predicted_claims": [
            np.sum(weight_test.to_numpy() * glm_test_prediction),
            np.sum(weight_test.to_numpy() * lgbm_test_prediction),
        ],
    }
)
total_calibration["predicted_to_actual"] = total_calibration["predicted_claims"] / total_calibration["actual_claims"]
glm_calibration = calibration_deciles("GLM", test["ClaimNb"].to_numpy(), weight_test.to_numpy(), glm_test_prediction)
lgbm_calibration = calibration_deciles("LightGBM", test["ClaimNb"].to_numpy(), weight_test.to_numpy(), lgbm_test_prediction)
calibration_by_decile = pd.concat([glm_calibration, lgbm_calibration], ignore_index=True)
perfect_score = y_test.to_numpy()
perfect_gini = weighted_gini(test["ClaimNb"], weight_test, perfect_score)
glm_gini = weighted_gini(test["ClaimNb"], weight_test, glm_test_prediction)
lgbm_gini = weighted_gini(test["ClaimNb"], weight_test, lgbm_test_prediction)
ranking_table = pd.DataFrame(
    {
        "model": ["GLM", "LightGBM"],
        "top_decile_lift": [
            glm_calibration.loc[glm_calibration["risk_decile"].eq(10), "lift"].iloc[0],
            lgbm_calibration.loc[lgbm_calibration["risk_decile"].eq(10), "lift"].iloc[0],
        ],
        "gini": [glm_gini, lgbm_gini],
        "normalized_gini": [glm_gini / perfect_gini, lgbm_gini / perfect_gini],
    }
)
display(deviance_table)
display(total_calibration)
display(calibration_by_decile)
display(ranking_table)


,model,train_mean_poisson_deviance,test_mean_poisson_deviance
0,Baseline,0.626328,0.625530
1,GLM,0.592741,0.592996
2,LightGBM,0.553675,0.570219


,model,actual_claims,predicted_claims,predicted_to_actual
0,GLM,7236,7221.220552,0.997958
1,LightGBM,7236,7230.773845,0.999278


,model,risk_decile,exposure,actual_claims,predicted_claims,observed_frequency,predicted_frequency,lift
0,GLM,1,7161.294644,292,328.767855,0.040775,0.045909,0.403533
1,GLM,2,7161.224709,388,416.211534,0.054181,0.058120,0.536207
2,GLM,3,7161.331994,523,469.781434,0.073031,0.065600,0.722763
3,GLM,4,7160.588774,546,513.230601,0.076251,0.071674,0.754626
4,GLM,5,7161.554360,547,554.815399,0.076380,0.077471,0.755907
5,GLM,6,7161.410662,594,600.237480,0.082945,0.083816,0.820873
6,GLM,7,7161.157200,666,659.574980,0.093002,0.092105,0.920405
7,GLM,8,7161.108868,781,764.311044,0.109061,0.106731,1.079341
8,GLM,9,7161.160693,1012,1000.080995,0.141318,0.139653,1.398573
9,GLM,10,7161.310737,1887,1914.209230,0.263499,0.267299,2.607759


,model,top_decile_lift,gini,normalized_gini
0,GLM,2.607759,0.289518,0.302167
1,LightGBM,3.126144,0.339571,0.354408


The GLM reduces test deviance by 5.2% versus baseline; LightGBM adds 3.8%, for 8.8% overall. Total predicted claims are 0.20% low for the GLM and 0.07% low for LightGBM. Normalised Gini is 0.302 and 0.354 respectively; Gini measures ranking, not calibration.

In [10]:
import joblib

glm_pipeline_path = Path("data") / "glm_pipeline.joblib"
lgbm_pipeline_path = Path("data") / "lgbm_pipeline.joblib"
test_predictions_path = Path("data") / "test_predictions.parquet"

joblib.dump(glm_model, glm_pipeline_path)
joblib.dump(lgbm_model, lgbm_pipeline_path)

test_predictions = test[["IDpol", "ClaimNb", "Exposure"] + feature_columns].copy()
test_predictions["glm_predicted_rate"] = glm_test_prediction
test_predictions["lgbm_predicted_rate"] = lgbm_test_prediction
test_predictions["baseline_rate"] = baseline_rate
test_predictions.to_parquet(test_predictions_path)

print(glm_pipeline_path, glm_pipeline_path.stat().st_size)
print(lgbm_pipeline_path, lgbm_pipeline_path.stat().st_size)
print(test_predictions_path, test_predictions.shape)
print(list(test_predictions.columns))

glm_reloaded = joblib.load(glm_pipeline_path)
lgbm_reloaded = joblib.load(lgbm_pipeline_path)
test_predictions_reloaded = pd.read_parquet(test_predictions_path)
print("glm reload matches", np.allclose(glm_reloaded.predict(X_test), glm_test_prediction))
print("lgbm reload matches", np.allclose(lgbm_reloaded.predict(X_test), lgbm_test_prediction))
print("parquet round trip", test_predictions_reloaded.equals(test_predictions))
print("baseline_rate", test_predictions_reloaded["baseline_rate"].iloc[0])

data\glm_pipeline.joblib 8866
data\lgbm_pipeline.joblib 2120928
data\test_predictions.parquet (135603, 15)
['IDpol', 'ClaimNb', 'Exposure', 'Area', 'VehPower', 'VehAge', 'DrivAge', 'BonusMalus', 'VehBrand', 'VehGas', 'Density', 'Region', 'glm_predicted_rate', 'lgbm_predicted_rate', 'baseline_rate']


glm reload matches True


lgbm reload matches True
parquet round trip True
baseline_rate 0.10061790715750009
